# 04 - 3DGS Training & Evaluation (Paper-Quality)

Trains per-volume 3DGS models on ALL test cases with multiple sparse ratios.

## Pipeline
1. For each test case and each R:
   - Initialize axis-aligned Gaussians
   - Optimize with LR scheduling + adaptive densification
   - Evaluate on held-out target slices
2. Report PSNR, SSIM, MAE, ROI metrics
3. Record training and inference times

## Key improvements over demo version
- 3000 iterations (vs 300 in demo)
- Position LR exponential decay (standard 3DGS)
- Periodic opacity reset for better pruning
- Edge-aware adaptive initialization option
- All test cases evaluated
- Multiple sparse ratios (R=2, R=3, R=4)

In [ ]:
# ========================== CONFIGURATION ==========================
FULL_EXPERIMENT = True
SPARSE_RATIOS = [2, 3, 4]
SEED = 42

if not FULL_EXPERIMENT:
    NUM_ITERATIONS = 500
    MAX_TEST_CASES = 2
    SPARSE_RATIOS = [2, 3]
else:
    NUM_ITERATIONS = 3000
    MAX_TEST_CASES = None  # All
# ===================================================================

In [ ]:
import os, sys, json, subprocess, time, gc
import numpy as np
import torch
from pathlib import Path
from copy import deepcopy

WORK_DIR = Path("/kaggle/working")
OUTPUT_ROOT = WORK_DIR / "outputs"
CHECKPOINT_ROOT = WORK_DIR / "checkpoints"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/PhoenixEvo/ct-slice-interpolation-3dgs.git"
REPO_DIR = WORK_DIR / "ct-slice-interpolation-3dgs"
if not (REPO_DIR / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from src.utils.seed import set_seed
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Load split and config
prepared_dir = OUTPUT_ROOT / "prepared"
with open(prepared_dir / "split_info.json") as f:
    split_info = json.load(f)

from src.utils.config import load_config, update_config
config = load_config("configs/default.yaml")
config = update_config(config, {
    "data": {"output_root": str(OUTPUT_ROOT)},
})

test_cases = split_info["test"]
if MAX_TEST_CASES is not None:
    test_cases = test_cases[:MAX_TEST_CASES]

print(f"Test cases to process: {len(test_cases)}")
print(f"Sparse ratios: {SPARSE_RATIOS}")
print(f"Iterations per volume: {NUM_ITERATIONS}")
print(f"Total experiments: {len(test_cases) * len(SPARSE_RATIOS)}")

In [ ]:
from tqdm import tqdm
from src.training.trainer_3dgs import Trainer3DGS
from src.data.sparse_simulator import SparseSimulator

organ_labels = {"liver": 1, "bladder": 2, "lungs": 3, "kidneys": 4, "bone": 5}
all_3dgs_results = []

for R in SPARSE_RATIOS:
    print(f"\n{'='*60}")
    print(f"  3DGS Training - Sparse Ratio R={R}")
    print(f"{'='*60}")

    out_dir = OUTPUT_ROOT / "3dgs" / f"R{R}"
    out_dir.mkdir(parents=True, exist_ok=True)

    for case_idx in tqdm(test_cases, desc=f"R={R}"):
        vol_path = prepared_dir / f"volume_case{case_idx}.npy"
        if not vol_path.exists():
            print(f"  Skipping case {case_idx}: not prepared")
            continue

        volume = np.load(vol_path)
        labels = None
        lab_path = prepared_dir / f"labels_case{case_idx}.npy"
        if lab_path.exists():
            labels = np.load(lab_path)

        sim = SparseSimulator(sparse_ratio=R)
        sparse = sim.simulate(volume)

        # Configure for this run
        cfg = deepcopy(config)
        cfg["gaussian"]["num_iterations"] = NUM_ITERATIONS

        set_seed(SEED)

        ckpt_dir = str(CHECKPOINT_ROOT / "3dgs" / f"case{case_idx}_R{R}")

        print(f"\n  Case {case_idx} | Volume {volume.shape} | "
              f"Observed: {len(sparse['observed_indices'])} | "
              f"Targets: {len(sparse['target_indices'])}")

        trainer = Trainer3DGS(
            volume=volume,
            observed_indices=sparse["observed_indices"],
            target_indices=sparse["target_indices"],
            config=cfg,
            labels=labels,
            device=device,
            checkpoint_dir=ckpt_dir,
        )

        history = trainer.train()

        # Evaluate
        eval_result = trainer.evaluate_on_targets(organ_labels=organ_labels)
        s = eval_result["summary"]
        print(f"  Results: PSNR={s['mean_psnr']:.2f} | SSIM={s['mean_ssim']:.4f} | "
              f"MAE={s.get('mae', 0):.4f} | #GS={s.get('num_gaussians_final', 0)} | "
              f"Train: {s.get('training_time_s', 0):.1f}s | Infer: {s.get('inference_time_s', 0):.1f}s")

        all_3dgs_results.append({
            "method": "3dgs",
            "case_idx": int(case_idx),
            "sparse_ratio": R,
            **s,
        })

        # Save per-case results
        with open(out_dir / f"case{case_idx}.json", "w") as f:
            json.dump(eval_result, f, indent=2, default=float)

        # Free GPU memory
        del trainer
        torch.cuda.empty_cache()
        gc.collect()

print(f"\nTotal 3DGS evaluations: {len(all_3dgs_results)}")

In [ ]:
import pandas as pd

df_3dgs = pd.DataFrame(all_3dgs_results)

print("\n=== 3DGS - Summary Table ===")
summary = df_3dgs.groupby(["sparse_ratio"]).agg({
    "mean_psnr": ["mean", "std"],
    "mean_ssim": ["mean", "std"],
    "mean_mae": ["mean", "std"] if "mean_mae" in df_3dgs.columns else ["mean"],
    "training_time_s": ["mean"],
    "inference_time_s": ["mean"],
    "num_gaussians_final": ["mean"],
}).round(4)
print(summary.to_string())

# Save
df_3dgs.to_csv(OUTPUT_ROOT / "3dgs" / "summary.csv", index=False)
print("\nSaved to:", OUTPUT_ROOT / "3dgs" / "summary.csv")